# Thử nghiệm: gộp Direction của XGBoost và LSTM (touch giữ XGBoost)

**Giả thuyết:** Direction LSTM mang thông tin hướng khác với XGBoost (`compare_models`: touch XGB + Direction LSTM có TB seed 0.180 nhưng std 0.059). Hai model sai theo cách khác nhau, nên gộp lại có thể giữ TB seed cao mà thu hẹp dải seed.

**Cách gộp điểm hướng d = p_up − 0.5**, trộn theo từng seed: XGB seed s với LSTM seed s.
- `raw`: d = w·d_XGB + (1 − w)·d_LSTM.
- `z`: mỗi model chia d cho độ lệch chuẩn của chính nó trong 90 ngày trước (chỉ nhìn quá khứ), rồi mới trộn. Cách này giúp model có biên độ dự báo lớn hơn không lấn át model kia.
- w ∈ {1, 0.75, 0.5, 0.25, 0}. Thêm TabM vào gộp 3 model với trọng số bằng nhau, để tham chiếu.

**Giữ nguyên:** touch XGBoost, tham số giao dịch đã khoá (touch 20%, Direction 10%, …), luật lệnh.

**Chọn:** theo **dev** 2022–2024 (TB seed, seed tệ nhất); test 2025–2026 chỉ để xác nhận. Đạt khi TB seed test tăng VÀ dải seed hẹp lại so với XGBoost.

**Cần chạy `compare_models.ipynb` trước.** Notebook này chỉ dùng dự báo đã cache, không train thêm. Kết quả lưu ở `artifacts/1h_strategy_v2/exp_direction_blend/`.

In [1]:
import os, sys
from dataclasses import replace
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "model_training").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "model_training"))
import numpy as np
import pandas as pd
import strategy_h1 as st
import futures_paper as fu

pd.set_option("display.width", 240, "display.max_columns", 50)
cfg = st.load_locked("artifacts/1h_strategy_v2/config_locked.json")
OUT = Path(cfg.output_dir) / "exp_direction_blend"
OUT.mkdir(parents=True, exist_ok=True)
D = st.build(cfg)
P = {k: {ph: st.walk_forward(D, replace(cfg, model_kind=k), ph)[0] for ph in ("dev", "test")}   # từ cache
     for k in ("xgb", "lstm", "tabm")}
SEEDS = [c[len("p_up_s"):] for c in P["xgb"]["test"].columns if c.startswith("p_up_s")]
for ph in ("dev", "test"):
    for k in ("lstm", "tabm"):
        assert np.array_equal(P[k][ph].t_ns.to_numpy(), P["xgb"][ph].t_ns.to_numpy())

Dữ liệu: 54,979 nến 1h, 53,522 mẫu (2020-08-30 -> 2026-10-08), touch 92 feature, direction 37 feature, 12s
dev 2022Q1: AUC touch 0.610 | AUC direction 0.616 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.695 | AUC direction 0.536 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.605 | AUC direction 0.520 | train 11,719 | 0s
dev 2022Q4: AUC touch 0.689 | AUC direction 0.601 | train 13,879 | 0s
dev 2023Q1: AUC touch 0.664 | AUC direction 0.675 | train 16,063 | 0s
dev 2023Q2: AUC touch 0.692 | AUC direction 0.588 | train 18,271 | 0s
dev 2023Q3: AUC touch 0.751 | AUC direction 0.565 | train 20,479 | 0s
dev 2023Q4: AUC touch 0.721 | AUC direction 0.566 | train 22,639 | 0s
dev 2024Q1: AUC touch 0.718 | AUC direction 0.571 | train 24,823 | 0s
dev 2024Q2: AUC touch 0.749 | AUC direction 0.587 | train 27,031 | 0s
dev 2024Q3: AUC touch 0.735 | AUC direction 0.585 | train 29,239 | 0s
dev 2024Q4: AUC touch 0.778 | AUC direction 0.626 | train 31,423 | 0s
test 2025Q1: AUC touch 0.763 | AUC direction 0.552 | tr

In [2]:
def zscore(P, col, days=90, min_hours=24):
    """d / std(d trên các nến [t - days, t) của cùng fold); chỉ nhìn quá khứ. Nến đầu fold chưa đủ lịch sử -> 0."""
    d = (P[col].to_numpy(np.float64) - 0.5)
    out = np.zeros(len(P))
    for _, idx in P.groupby("fold", sort=False).indices.items():
        s = pd.Series(d[idx], index=pd.to_datetime(P.t_ns.to_numpy()[idx], utc=True))
        sd = s.rolling(f"{days}D", min_periods=min_hours).std().shift(1)
        z = (s / sd).to_numpy()
        out[idx] = np.where(np.isfinite(z), z, 0.0)
    return out

def blend(ph, weights: dict, mode: str):
    """P của XGB (touch giữ nguyên) với p_up (ensemble + từng seed) thay bằng điểm gộp."""
    Q = P["xgb"][ph].copy()
    for suf in [""] + [f"_s{s}" for s in SEEDS]:
        if mode == "raw":
            d = sum(w * (P[k][ph][f"p_up{suf}"].to_numpy(np.float64) - 0.5) for k, w in weights.items())
            Q[f"p_up{suf}"] = 0.5 + d
        else:
            d = sum(w * zscore(P[k][ph], f"p_up{suf}") for k, w in weights.items())
            Q[f"p_up{suf}"] = 0.5 + 0.1 * d                 # thang không ảnh hưởng ngưỡng phân vị
    return Q

VARIANTS = {}
for mode in ("raw", "z"):
    for w in (1.0, 0.75, 0.5, 0.25, 0.0):
        VARIANTS[f"{mode} | XGB {w:.2f} + LSTM {1 - w:.2f}"] = ({"xgb": w, "lstm": 1 - w}, mode)
    VARIANTS[f"{mode} | XGB + LSTM + TabM (1/3)"] = ({"xgb": 1 / 3, "lstm": 1 / 3, "tabm": 1 / 3}, mode)
len(VARIANTS)

12

## 1. AUC Direction của điểm gộp

In [3]:
rows = []
for name, (w, mode) in VARIANTS.items():
    for ph in ("dev", "test"):
        a = st.direction_auc(D, blend(ph, w, mode))
        rows.append({"bản": name, "phase": ph, "AUC chạm trước": a["AUC chạm trước (đã chạm)"].mean(),
                     "quý kém nhất": a["AUC chạm trước (đã chạm)"].min()})
AUC = pd.DataFrame(rows).pivot(index="bản", columns="phase", values=["AUC chạm trước", "quý kém nhất"])
AUC.to_csv(OUT / "auc.csv")
AUC.round(4)

AUC chạm trước         quý kém nhất        
phase                                    dev    test          dev    test
bản                                                                      
raw | XGB + LSTM + TabM (1/3)         0.5977  0.5726       0.5257  0.5249
raw | XGB 0.00 + LSTM 1.00            0.5920  0.5625       0.5249  0.4891
raw | XGB 0.25 + LSTM 0.75            0.5968  0.5677       0.5225  0.5080
raw | XGB 0.50 + LSTM 0.50            0.5965  0.5696       0.5241  0.5290
raw | XGB 0.75 + LSTM 0.25            0.5928  0.5699       0.5218  0.5298
raw | XGB 1.00 + LSTM 0.00            0.5862  0.5696       0.5195  0.5304
z | XGB + LSTM + TabM (1/3)           0.5983  0.5733       0.5243  0.5223
z | XGB 0.00 + LSTM 1.00              0.5931  0.5633       0.5244  0.4893
z | XGB 0.25 + LSTM 0.75              0.5969  0.5683       0.5215  0.5059
z | XGB 0.50 + LSTM 0.50              0.5972  0.5703       0.5223  0.5269
z | XGB 0.75 + LSTM 0.25              0.5929  0.5710       0.5198  0.5302
z | XGB 1.00 + LSTM 0.00              0.5856  0.5704       0.5175  0.5317

## 2. Giao dịch với tham số đã khoá: dev và test, từng seed

In [4]:
def row(name, Pd, Pt):
    out = {"bản": name}
    for ph, PP in (("dev", Pd), ("test", Pt)):
        ev = st.evaluate(D, PP, cfg, ph)
        S = st.seed_table(D, PP, cfg, ph)
        s = ev["summary"]
        out.update({f"{ph} lệnh": s["trades"], f"{ph} R ens": s["R_per_trade"],
                    f"{ph} TB seed": S.loc["TB seed", "R/lệnh"], f"{ph} std seed": S.loc["std seed", "R/lệnh"],
                    f"{ph} seed tệ nhất": S.loc["seed tệ nhất", "R/lệnh"], f"{ph} DD %": s["max_dd_pct"],
                    f"{ph} lãi % vốn": (s["equity_multiple"] - 1) * 100})
        out.update({f"{ph} R {y}": s["R_by_year"][y] / s["trades_by_year"][y] for y in sorted(s["trades_by_year"])})
        if ph == "test":
            out["test quý âm"] = s["negative_quarters"]
            out["test LONG"] = s["long_share"]
    return out

T = pd.DataFrame([row(n, blend("dev", w, m), blend("test", w, m)) for n, (w, m) in VARIANTS.items()]).set_index("bản")
T["dev điểm"] = 0.5 * T["dev TB seed"] + 0.5 * T["dev seed tệ nhất"]   # chọn theo dev: TB seed và seed tệ nhất
T = T.sort_values("dev điểm", ascending=False)
T.to_csv(OUT / "blend_results.csv")
show = ["dev điểm", "dev TB seed", "dev std seed", "dev seed tệ nhất", "test lệnh", "test R ens", "test TB seed",
        "test std seed", "test seed tệ nhất", "test DD %", "test lãi % vốn", "test quý âm", "test LONG"]
T[show].round(3)

,dev điểm,dev TB seed,dev std seed,dev seed tệ nhất,test lệnh,test R ens,test TB seed,test std seed,test seed tệ nhất,test DD %,test lãi % vốn,test quý âm,test LONG
bản,,,,,,,,,,,,,
z | XGB + LSTM + TabM (1/3),0.195,0.205,0.022,0.186,390,0.180,0.164,0.024,0.130,12.660,93.017,2,0.505
raw | XGB 0.50 + LSTM 0.50,0.186,0.196,0.021,0.176,376,0.214,0.194,0.020,0.174,9.247,114.778,1,0.492
raw | XGB + LSTM + TabM (1/3),0.182,0.197,0.021,0.167,379,0.193,0.180,0.009,0.173,9.247,100.054,3,0.530
raw | XGB 0.75 + LSTM 0.25,0.181,0.192,0.020,0.170,382,0.184,0.180,0.026,0.138,10.639,94.149,1,0.505
z | XGB 0.50 + LSTM 0.50,0.181,0.202,0.030,0.160,385,0.177,0.174,0.026,0.150,12.863,89.737,1,0.486
z | XGB 0.75 + LSTM 0.25,0.179,0.191,0.017,0.166,397,0.176,0.164,0.034,0.113,12.850,92.700,1,0.486
z | XGB 1.00 + LSTM 0.00,0.166,0.177,0.030,0.156,407,0.149,0.137,0.014,0.128,10.090,76.371,1,0.491
raw | XGB 0.25 + LSTM 0.75,0.163,0.177,0.030,0.150,371,0.209,0.194,0.038,0.145,15.860,108.024,2,0.507
raw | XGB 1.00 + LSTM 0.00,0.155,0.166,0.022,0.145,400,0.168,0.155,0.025,0.133,11.111,87.978,1,0.500


In [5]:
# R/lệnh theo năm (ensemble)
T[[c for c in T.columns if " R 20" in c]].round(3)

,dev R 2022,dev R 2023,dev R 2024,test R 2025,test R 2026
bản,,,,,
z | XGB + LSTM + TabM (1/3),-0.014,0.373,0.250,0.138,0.223
raw | XGB 0.50 + LSTM 0.50,0.039,0.353,0.233,0.178,0.254
raw | XGB + LSTM + TabM (1/3),0.024,0.348,0.245,0.136,0.256
raw | XGB 0.75 + LSTM 0.25,0.007,0.329,0.257,0.173,0.197
z | XGB 0.50 + LSTM 0.50,0.028,0.328,0.245,0.129,0.226
z | XGB 0.75 + LSTM 0.25,-0.009,0.364,0.230,0.166,0.186
z | XGB 1.00 + LSTM 0.00,0.028,0.376,0.170,0.117,0.187
raw | XGB 0.25 + LSTM 0.75,0.056,0.303,0.176,0.124,0.293
raw | XGB 1.00 + LSTM 0.00,-0.013,0.356,0.179,0.150,0.190


## 3. So sánh với XGBoost: bản dev chọn + 2 bản trộn 50/50

In [6]:
best = T.index[0]
print("dev chọn:", best)
base_name = "raw | XGB 1.00 + LSTM 0.00"
pick = list(dict.fromkeys([base_name, best, "raw | XGB 0.50 + LSTM 0.50", "z | XGB 0.50 + LSTM 0.50"]))
for n in pick:
    w, m = VARIANTS[n]
    ev = st.evaluate(D, blend("test", w, m), cfg, "test")
    print("=====", n)
    display(ev["by_year"][["lệnh", "R/lệnh", "thắng", "tỉ lệ LONG", "hết giờ", "lãi % vốn"]])
pd.DataFrame({n: st.evaluate(D, blend("test", *VARIANTS[n]), cfg, "test")["by_quarter"]["R/lệnh"] for n in pick}).round(3)

dev chọn: z | XGB + LSTM + TabM (1/3)
===== raw | XGB 1.00 + LSTM 0.00


,lệnh,R/lệnh,thắng,tỉ lệ LONG,hết giờ,lãi % vốn
năm,,,,,,
2025,224,0.15,0.594,0.406,0.201,36.635
2026,176,0.19,0.574,0.619,0.295,37.577


===== z | XGB + LSTM + TabM (1/3)


,lệnh,R/lệnh,thắng,tỉ lệ LONG,hết giờ,lãi % vốn
năm,,,,,,
2025,199,0.138,0.583,0.442,0.201,28.526
2026,191,0.223,0.581,0.571,0.277,50.178


===== raw | XGB 0.50 + LSTM 0.50


,lệnh,R/lệnh,thắng,tỉ lệ LONG,hết giờ,lãi % vốn
năm,,,,,,
2025,199,0.178,0.613,0.412,0.181,39.400
2026,177,0.254,0.616,0.582,0.266,54.073


===== z | XGB 0.50 + LSTM 0.50


,lệnh,R/lệnh,thắng,tỉ lệ LONG,hết giờ,lãi % vốn
năm,,,,,,
2025,192,0.129,0.578,0.411,0.188,25.225
2026,193,0.226,0.591,0.560,0.269,51.517


,raw | XGB 1.00 + LSTM 0.00,z | XGB + LSTM + TabM (1/3),raw | XGB 0.50 + LSTM 0.50,z | XGB 0.50 + LSTM 0.50
quý,,,,
2025Q1,0.057,0.083,0.011,0.085
2025Q2,0.324,0.335,0.411,0.323
2025Q3,-0.001,-0.032,-0.160,-0.155
2025Q4,0.211,0.164,0.350,0.271
2026Q1,0.188,0.282,0.274,0.284
2026Q2,0.109,0.184,0.228,0.171
2026Q3,0.381,0.252,0.303,0.234
2026Q4,0.930,-1.065,NaN,NaN


## 4. Paper futures cross x50, margin 1% vốn (test)

In [7]:
funding = pd.read_csv(cfg.funding_csv)
fp = fu.FuturesParams(capital=100_000, leverage=50, margin_frac=0.01, compound=True, margin_mode="cross")
rows = []
for n in pick:
    sg = st.evaluate(D, blend("test", *VARIANTS[n]), cfg, "test")["signals"]
    r = fu.run(D, sg[sg.signal], cfg, fp, funding)
    s, y = r["summary"], fu.by_period(r, "Y")["lãi % vốn"]
    rows.append({"bản": n, "lệnh": s["lệnh"], "lãi %": s["lãi %"], "CAGR %": s["lãi %/năm (CAGR)"], "DD %": s["DD max %"],
                 "thắng": s["thắng"], **{f"lãi % {k}": v for k, v in y.items()}})
F = pd.DataFrame(rows).set_index("bản")
F.to_csv(OUT / "futures_x50.csv")
F.round(2)

,lệnh,lãi %,CAGR %,DD %,thắng,lãi % 2025,lãi % 2026
bản,,,,,,,
raw | XGB 1.00 + LSTM 0.00,400,55.18,28.68,7.42,0.58,20.67,28.60
z | XGB + LSTM + TabM (1/3),390,66.09,34.05,7.54,0.58,17.81,40.98
raw | XGB 0.50 + LSTM 0.50,376,74.73,38.67,6.78,0.61,25.39,39.35
z | XGB 0.50 + LSTM 0.50,385,66.33,34.73,7.34,0.58,17.53,41.52
